# LOBO (leave-one-batch-out) validation

Thin runner. All logic lives in `pipeline/lobo_validation.py`; it reads the per-batch LOBO outputs in `config.MODELING_DIR / "LOBO_Results" / <batch_id>/` (produced by `run_lobo_validation.py` + `compute_lobo_ood.py`) and writes `LOBO_Results/mmd_summary.csv` + `LOBO_Results/mmd_summary.png`.

Replaces the discontinued classifier-based `discrimination_control` (see memory `project_hc_calibration_batch_confound` / `project_lobo_validation_design`). For each Tier-A batch (HC and disease co-located in the same batch) with a stably-estimable held-out-HC noise floor (`n_hc >= pipeline.lobo_validation.MIN_N_HC`), tests whether held-out-HC and held-out-disease Z-vectors differ (MMD^2, RBF kernel over all genes jointly, permutation p-value), and reports the direction: is disease farther from the in-fold HC reference centroid than the held-out-HC noise floor is.

In [45]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [46]:
import sys, warnings
warnings.filterwarnings('ignore')
sys.path.insert(0, '.')
sys.path.insert(0, '..')
from pipeline import lobo_validation as lv


In [ ]:
df = lv.mmd_summary_cached(force=False)
print(df.to_string(index=False))
fig_bar, fig_direction = lv.plot_mmd_summary(df, fig_dir=lv.LOBO_DIR)
n_sig_and_farther = ((df['perm_p'] < 0.05) & df['disease_farther']).sum()
print(str(n_sig_and_farther) + '/' + str(len(df)) + ' batches: MMD significant (p<0.05) AND disease farther from the in-fold HC reference than the held-out-HC noise floor.')